# Bài 1 · Mô hình trò chơi Tic-Tac-Toe

Trạng thái bàn cờ là danh sách 9 phần tử thuộc `{' ', 'x', 'o'}` theo thứ tự từ trái sang phải, từ trên xuống dưới. X đi trước. Lượt đi được truyền vào hàm chuyển trạng thái và đổi sau mỗi nước.

| Thành phần | Cách biểu diễn |
|---|---|
| `actions(s)` | Chỉ số các ô trống từ 0 đến 8. Vòng chơi chỉ gọi tác tử khi ván chưa kết thúc. |
| `result(s, player, a)` | Tạo bản sao của bàn rồi đặt quân của `player` vào ô trống `a`. |
| `terminal(s)` | Kiểm tra hàng, cột, hai đường chéo và tình trạng hết ô trống. |
| `utility(s, player)` | +1 nếu `player` thắng, 0 nếu hòa, −1 nếu thua. Ván đang chơi trả về `None`. Mặc định Max là X. |
| `check_board(s)` | Trả về `'x'`, `'o'`, `'d'` (hòa) hoặc `'n'` (còn lượt). |

In [1]:
from tictactoe import (
    actions, check_board, empty_board, play, random_player, result,
    show_board, terminal, utility,
)

board = empty_board()
show_board(board)
print("Hành động ban đầu:", actions(board))
after = result(board, "x", 4)
show_board(after)
print("Sau khi X đi ô 4:", actions(after), check_board(after),
      terminal(after), utility(after))

win = list("xo xo x  ")
draw = list("xoxxoooxx")
show_board(win)
for label, state in [("X thắng", win), ("Hòa", draw)]:
    print(label, check_board(state), terminal(state), utility(state), utility(state, "o"))

assert board == empty_board() and after[4] == "x" and 4 not in actions(after)
assert not terminal(after) and utility(after) is None
assert terminal(win) and check_board(win) == "x" and utility(win, "o") == -1
assert terminal(draw) and check_board(draw) == "d" and utility(draw) == 0

0 | 1 | 2
---------
3 | 4 | 5
---------
6 | 7 | 8
Hành động ban đầu: [0, 1, 2, 3, 4, 5, 6, 7, 8]
0 | 1 | 2
---------
3 | x | 5
---------
6 | 7 | 8
Sau khi X đi ô 4: [0, 1, 2, 3, 5, 6, 7, 8] n False None
x | o | 2
---------
x | o | 5
---------
x | 7 | 8
X thắng x True 1 -1
Hòa d True 0 0


Ô 4 là tâm. Sau khi X đánh ô này, bàn đầu vào vẫn rỗng và nước 4 biến mất khỏi tập hành động của bàn mới. Hàng dọc 0–3–6 kết thúc ván ở ví dụ X thắng.

## Hai tác tử Random trong 1.000 ván

Đặt seed NumPy là 51000. Cột thời gian đo riêng lượt của X bằng `perf_counter()`; vòng chơi và lượt của O nằm ngoài phép đo.

In [2]:
from time import perf_counter
import numpy as np

def random_baseline(games=1000):
    elapsed = moves = 0

    def timed_random(board, player):
        nonlocal elapsed, moves
        start = perf_counter()
        move = random_player(board, player)
        elapsed += perf_counter() - start
        moves += 1
        return move

    np.random.seed(51000)
    outcomes = play(timed_random, random_player, N=games)
    assert sum(outcomes.values()) == games
    return dict(games=games, wins=outcomes["x"], draws=outcomes["d"], losses=outcomes["o"],
                ms_per_move=1000 * elapsed / moves)

stats = random_baseline()
for label, key in [("X thắng", "wins"), ("O thắng", "losses"), ("Hòa", "draws")]:
    print(f"{label:8} {stats[key]:4d} ván ({stats[key] / stats['games']:.1%})")
print(f"Random X: {stats['ms_per_move']:.4f} ms/nước")

X thắng   584 ván (58.4%)
O thắng   303 ván (30.3%)
Hòa       113 ván (11.3%)
Random X: 0.0218 ms/nước


Với hai tác tử chọn nước đồng đều, X thắng nhiều ván hơn O trong mẫu này. X có lượt đặt quân đầu tiên và có thể hoàn tất ba quân trước O một lượt. Kết quả giữa hai người chơi ngẫu nhiên mô tả lợi thế đi trước trong mẫu; giá trị bàn trống với hai bên chơi tối ưu được kiểm tra ở Bài 3.